# Comprehensive Exploratory Data Analysis of Human Trafficking Reports to the Disque 100 Hotline

## Step 8: Multivariate Exploration

This notebook integrates complaint, violation, victim, suspect, relationship, temporal, and geographic attributes to explore multidimensional patterns in recorded Brazilian human rights hotline data.

### Objectives

1. Build a defensible multivariate analytical table.
2. Quantify associations among categorical variables with bias-corrected Cramér's V.
3. Visualize multi-way combinations using faceted proportions and parallel categories.
4. Use Multiple Correspondence Analysis (MCA) for exploratory categorical ordination.
5. Explore record segments using cluster analysis on low-dimensional MCA coordinates.
6. Profile clusters with interpretable category lifts and temporal/geographic context.
7. Evaluate stability, missingness sensitivity, sparse categories, and disclosure risk.
8. Export reproducible tables, figures, assignments, and findings.

> MCA and clustering are exploratory tools. Components and clusters are not latent truths, victim typologies, causal mechanisms, or risk classes. They summarize patterns in the recorded administrative data and may reflect reporting, form design, coding, missingness, and repeated complaint structure.


## 8.1 Imports and configuration

The workflow uses `pandas`, `numpy`, `scipy`, `scikit-learn`, and Plotly. MCA is implemented from an indicator matrix using correspondence-analysis scaling, avoiding dependence on a specialized package. Clustering uses K-means on MCA coordinates, not raw one-hot columns.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.extmath import randomized_svd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_colwidth", 180)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_08_multivariate_exploration")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTING_MODE = "record"  # record or complaint
IDENTIFIER = "hash"
DATE_COLUMN = "data_de_cadastro"
MISSING_LABEL = "<MISSING>"
OTHER_LABEL = "OTHER"
MIN_CATEGORY_COUNT = 20
MAX_LEVELS_PER_VARIABLE = 15
MAX_MCA_COMPONENTS = 10
CLUSTER_K_RANGE = range(2, 9)
FINAL_N_CLUSTERS = None  # None selects the best silhouette result
RANDOM_STATE = 42
SAMPLE_FOR_PLOTS = 5000
SUPPRESSION_THRESHOLD = 5
EXPORT_FIGURES_HTML = True
PLOTLY_TEMPLATE = "plotly_white"


## 8.2 Load the cleaned dataset and establish the analytical unit

Complaint mode keeps the earliest record per non-null `hash`. Use it only if Steps 1 through 5 established that a first-record representation is defensible. Record mode is the conservative default for violation-level exploration.


In [2]:
def load_clean_dataset():
    if PARQUET_PATH.exists():
        return pd.read_parquet(PARQUET_PATH), PARQUET_PATH
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, low_memory=False), CSV_PATH
    raise FileNotFoundError("Run Step 3 or update PARQUET_PATH/CSV_PATH.")


df, DATA_SOURCE = load_clean_dataset()
if DATE_COLUMN in df.columns:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint":
    if IDENTIFIER not in df.columns:
        raise ValueError("Complaint mode requires hash.")
    df_analysis = df.dropna(subset=[IDENTIFIER]).sort_values(DATE_COLUMN).drop_duplicates(IDENTIFIER).copy()
    unit_label = "complaints"
else:
    df_analysis = df.copy()
    unit_label = "records"

print("Source:", DATA_SOURCE.resolve())
print(f"Analytical population: {len(df_analysis):,} {unit_label}")


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Analytical population: 8,401 records


## 8.3 Select a substantively coherent feature set

The default feature set balances victim, suspect, complaint, relationship, temporal, and geography dimensions. Variables with extreme missingness, identifiers, free text, near uniqueness, or redundant geographic detail are excluded from the MCA and clustering table.


In [3]:
CANDIDATE_FEATURES = [
    "violacao", "grupo_vulneravel", "denuncia_emergencial",
    "canal_de_atendimento", "cenario_da_violacao", "frequencia",
    "relacao_vitima_suspeito", "genero_da_vitima",
    "faixa_etaria_da_vitima", "raca_cor_da_vitima",
    "orientacao_sexual_da_vitima", "deficiencia_da_vitima",
    "doenca_rara_da_vitima", "genero_do_suspeito",
    "faixa_etaria_do_suspeito", "raca_cor_do_suspeito",
    "natureza_juridica_do_suspeito", "suspeito_preso", "uf"
]

feature_inventory=[]
for column in CANDIDATE_FEATURES:
    if column not in df_analysis.columns:
        feature_inventory.append({"column":column,"present":False})
        continue
    s=df_analysis[column]
    feature_inventory.append({
        "column":column,"present":True,"missing_pct":round(s.isna().mean()*100,2),
        "unique_valid":int(s.nunique(dropna=True)),
        "top_category_pct_valid":round(s.value_counts(normalize=True,dropna=True).iloc[0]*100,2) if s.notna().any() else np.nan
    })
feature_inventory=pd.DataFrame(feature_inventory)
display(feature_inventory)

SELECTED_FEATURES=[
    c for c in CANDIDATE_FEATURES
    if c in df_analysis.columns
    and df_analysis[c].nunique(dropna=True) >= 2
    and df_analysis[c].isna().mean() < 0.80
]
print("Selected features:", SELECTED_FEATURES)


,column,present,missing_pct,unique_valid,top_category_pct_valid
0,violacao,True,2.26,100,6.52
1,grupo_vulneravel,True,0.00,9,61.92
2,denuncia_emergencial,True,0.01,4,91.02
3,canal_de_atendimento,True,0.00,8,72.50
4,cenario_da_violacao,True,0.00,28,21.14
5,frequencia,True,0.20,6,61.70
6,relacao_vitima_suspeito,True,0.35,39,44.51
7,genero_da_vitima,True,17.71,5,59.31
8,faixa_etaria_da_vitima,True,4.82,40,14.89
9,raca_cor_da_vitima,True,9.63,6,42.28


Selected features: ['violacao', 'grupo_vulneravel', 'denuncia_emergencial', 'canal_de_atendimento', 'cenario_da_violacao', 'frequencia', 'relacao_vitima_suspeito', 'genero_da_vitima', 'faixa_etaria_da_vitima', 'raca_cor_da_vitima', 'orientacao_sexual_da_vitima', 'genero_do_suspeito', 'faixa_etaria_do_suspeito', 'raca_cor_do_suspeito', 'natureza_juridica_do_suspeito', 'suspeito_preso', 'uf']


## 8.4 Prepare categorical features

Rare categories are grouped as `OTHER` only for multivariate exploration. Original values remain unchanged in `df_analysis`. Missingness is retained as an explicit category because complete-case analysis could discard a large and non-random portion of administrative records.


In [4]:
def collapse_categories(series, min_count=MIN_CATEGORY_COUNT, max_levels=MAX_LEVELS_PER_VARIABLE):
    s=series.astype("string").fillna(MISSING_LABEL)
    counts=s.value_counts(dropna=False)
    keep=set(counts[counts>=min_count].index)
    if len(keep)>max_levels-1:
        keep=set(counts.head(max_levels-1).index)
    return s.where(s.isin(keep),OTHER_LABEL)


X_cat=pd.DataFrame(index=df_analysis.index)
category_reduction_rows=[]
for column in SELECTED_FEATURES:
    before=df_analysis[column].astype("string").fillna(MISSING_LABEL)
    after=collapse_categories(df_analysis[column])
    X_cat[column]=after
    category_reduction_rows.append({
        "column":column,"levels_before":int(before.nunique()),
        "levels_after":int(after.nunique()),
        "records_grouped_as_other":int((after==OTHER_LABEL).sum()),
        "missing_category_records":int((after==MISSING_LABEL).sum())
    })
category_reduction=pd.DataFrame(category_reduction_rows)
display(category_reduction)
display(X_cat.head())


,column,levels_before,levels_after,records_grouped_as_other,missing_category_records
0,violacao,101,15,3363,190
1,grupo_vulneravel,9,9,16,0
2,denuncia_emergencial,5,4,6,0
3,canal_de_atendimento,8,7,15,0
4,cenario_da_violacao,28,15,252,0
5,frequencia,7,7,17,0
6,relacao_vitima_suspeito,40,15,485,0
7,genero_da_vitima,6,6,0,1488
8,faixa_etaria_da_vitima,41,15,2131,405
9,raca_cor_da_vitima,7,7,0,809


,violacao,grupo_vulneravel,denuncia_emergencial,canal_de_atendimento,cenario_da_violacao,frequencia,relacao_vitima_suspeito,genero_da_vitima,faixa_etaria_da_vitima,raca_cor_da_vitima,orientacao_sexual_da_vitima,genero_do_suspeito,faixa_etaria_do_suspeito,raca_cor_do_suspeito,natureza_juridica_do_suspeito,suspeito_preso,uf
0,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,OTHER,OTHER,VIA PÚBLICA,DIARIAMENTE,NÃO SE APLICA,FEMININO,<MISSING>,<MISSING>,<MISSING>,<MISSING>,<MISSING>,<MISSING>,COMUNIDADE,<MISSING>,RJ
1,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - NACIONAL,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NÃO,TELEFÔNICO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,SEMANALMENTE,MÃE,FEMININO,OTHER,BRANCA,<MISSING>,FEMININO,30 A 34 ANOS,BRANCA,PESSOA FÍSICA,<MISSING>,SP
2,<MISSING>,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NÃO,TELEFÔNICO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,SEMANALMENTE,MÃE,FEMININO,OTHER,BRANCA,<MISSING>,FEMININO,30 A 34 ANOS,BRANCA,PESSOA FÍSICA,<MISSING>,SP
3,OTHER,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NÃO,TELEFÔNICO,CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,SEMANALMENTE,MÃE,FEMININO,OTHER,BRANCA,<MISSING>,FEMININO,30 A 34 ANOS,BRANCA,PESSOA FÍSICA,<MISSING>,SP
4,LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE PESSOAS - INTERNACIONAL,VIOLÊNCIA CONTRA CRIANÇA OU ADOLESCENTE,NÃO,TELEFÔNICO,OUTROS,DIARIAMENTE,OUTROS,FEMININO,OTHER,BRANCA,<MISSING>,FEMININO,70 A 74 ANOS,PRETA,PESSOA FÍSICA,<MISSING>,SP


## 8.5 Pairwise categorical association matrix

Bias-corrected Cramér's V summarizes association strength between each pair of selected variables. It is not directional and does not imply causality. Strong associations may indicate conceptual overlap, form dependencies, or redundant inputs.


In [5]:
def cramers_v_bias_corrected(x,y):
    table=pd.crosstab(x,y)
    if table.shape[0]<2 or table.shape[1]<2: return np.nan
    chi2=chi2_contingency(table,correction=False)[0]
    n=table.to_numpy().sum()
    r,k=table.shape
    if n<=1: return np.nan
    phi2=chi2/n
    phi2corr=max(0,phi2-((k-1)*(r-1))/(n-1))
    rcorr=r-((r-1)**2)/(n-1)
    kcorr=k-((k-1)**2)/(n-1)
    denominator=min(kcorr-1,rcorr-1)
    return np.sqrt(phi2corr/denominator) if denominator>0 else np.nan

association_matrix=pd.DataFrame(np.eye(len(SELECTED_FEATURES)),index=SELECTED_FEATURES,columns=SELECTED_FEATURES)
for i,a in enumerate(SELECTED_FEATURES):
    for b in SELECTED_FEATURES[i+1:]:
        value=cramers_v_bias_corrected(X_cat[a],X_cat[b])
        association_matrix.loc[a,b]=value
        association_matrix.loc[b,a]=value

display(association_matrix.round(3))
fig=px.imshow(association_matrix.round(3),text_auto=".2f",aspect="auto",zmin=0,zmax=1,color_continuous_scale="Blues",title="Bias-corrected Cramér's V among selected features",labels={"color":"Cramér's V"})
fig.update_layout(template=PLOTLY_TEMPLATE,height=900)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"cramers_v_matrix.html",include_plotlyjs=True)


,violacao,grupo_vulneravel,denuncia_emergencial,canal_de_atendimento,cenario_da_violacao,frequencia,relacao_vitima_suspeito,genero_da_vitima,faixa_etaria_da_vitima,raca_cor_da_vitima,orientacao_sexual_da_vitima,genero_do_suspeito,faixa_etaria_do_suspeito,raca_cor_do_suspeito,natureza_juridica_do_suspeito,suspeito_preso,uf
violacao,1.000,0.129,0.086,0.110,0.104,0.076,0.093,0.113,0.131,0.168,0.114,0.068,0.048,0.077,0.071,0.081,0.090
grupo_vulneravel,0.129,1.000,0.109,0.186,0.236,0.133,0.246,0.293,0.396,0.213,0.320,0.150,0.137,0.135,0.141,0.110,0.214
denuncia_emergencial,0.086,0.109,1.000,0.086,0.288,0.226,0.273,0.122,0.159,0.204,0.112,0.078,0.135,0.079,0.115,0.114,0.157
canal_de_atendimento,0.110,0.186,0.086,1.000,0.232,0.169,0.195,0.200,0.251,0.223,0.157,0.176,0.177,0.158,0.245,0.145,0.284
cenario_da_violacao,0.104,0.236,0.288,0.232,1.000,0.295,0.334,0.272,0.269,0.327,0.208,0.237,0.179,0.263,0.253,0.195,0.332
frequencia,0.076,0.133,0.226,0.169,0.295,1.000,0.245,0.144,0.251,0.229,0.090,0.114,0.160,0.128,0.147,0.098,0.317
relacao_vitima_suspeito,0.093,0.246,0.273,0.195,0.334,0.245,1.000,0.235,0.280,0.311,0.290,0.244,0.228,0.251,0.285,0.217,0.269
genero_da_vitima,0.113,0.293,0.122,0.200,0.272,0.144,0.235,1.000,0.308,0.279,0.326,0.142,0.225,0.116,0.146,0.120,0.262
faixa_etaria_da_vitima,0.131,0.396,0.159,0.251,0.269,0.251,0.280,0.308,1.000,0.424,0.357,0.232,0.157,0.206,0.276,0.189,0.277
raca_cor_da_vitima,0.168,0.213,0.204,0.223,0.327,0.229,0.311,0.279,0.424,1.000,0.250,0.170,0.168,0.182,0.192,0.145,0.355


## 8.6 High-association and redundancy review

Pairs with Cramér's V of at least 0.50 are flagged for conceptual review. Removing one variable is not automatic; the decision depends on whether the relationship is expected, hierarchical, or caused by duplicated information.


In [6]:
upper=np.triu(np.ones(association_matrix.shape,dtype=bool),k=1)
high_association_pairs=(
    association_matrix.where(upper).stack().rename("cramers_v").reset_index()
)
high_association_pairs.columns=["variable_1","variable_2","cramers_v"]
high_association_pairs=high_association_pairs.sort_values("cramers_v",ascending=False)
display(high_association_pairs.head(50))


,variable_1,variable_2,cramers_v
217,faixa_etaria_do_suspeito,raca_cor_do_suspeito,0.465486
201,genero_do_suspeito,natureza_juridica_do_suspeito,0.441223
145,faixa_etaria_da_vitima,raca_cor_da_vitima,0.424133
219,faixa_etaria_do_suspeito,suspeito_preso,0.420248
25,grupo_vulneravel,faixa_etaria_da_vitima,0.395899
202,genero_do_suspeito,suspeito_preso,0.365321
146,faixa_etaria_da_vitima,orientacao_sexual_da_vitima,0.357174
169,raca_cor_da_vitima,uf,0.354786
236,raca_cor_do_suspeito,suspeito_preso,0.352126
199,genero_do_suspeito,faixa_etaria_do_suspeito,0.342119


## 8.7 Multi-way faceted proportions

These visualizations show how violation composition changes across victim age, victim gender, and relationship categories. Categories are collapsed only for readability, and proportions are calculated within each facet group.


In [7]:
if {"violacao","faixa_etaria_da_vitima","genero_da_vitima"}.issubset(X_cat.columns):
    temp=X_cat[["violacao","faixa_etaria_da_vitima","genero_da_vitima"]].copy()
    counts=temp.groupby(["faixa_etaria_da_vitima","genero_da_vitima","violacao"]).size().rename("count").reset_index()
    counts["group_total"]=counts.groupby(["faixa_etaria_da_vitima","genero_da_vitima"])["count"].transform("sum")
    counts["share_pct"]=counts["count"]/counts["group_total"]*100
    eligible=counts["group_total"]>=SUPPRESSION_THRESHOLD
    facet_violation=counts[eligible].copy()
    fig=px.bar(facet_violation,x="faixa_etaria_da_vitima",y="share_pct",color="violacao",facet_col="genero_da_vitima",facet_col_wrap=2,barmode="stack",title="Violation composition by victim age and gender",labels={"share_pct":"Within-group share (%)"})
    fig.update_layout(template=PLOTLY_TEMPLATE,height=900)
    fig.for_each_annotation(lambda a:a.update(text=a.text.split("=")[-1]))
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"faceted_violation_age_gender.html",include_plotlyjs=True)
else:
    facet_violation=pd.DataFrame()


In [8]:
if {"violacao","relacao_vitima_suspeito","grupo_vulneravel"}.issubset(X_cat.columns):
    temp=X_cat[["violacao","relacao_vitima_suspeito","grupo_vulneravel"]].copy()
    counts=temp.groupby(["grupo_vulneravel","relacao_vitima_suspeito","violacao"]).size().rename("count").reset_index()
    counts["group_total"]=counts.groupby(["grupo_vulneravel","relacao_vitima_suspeito"])["count"].transform("sum")
    counts["share_pct"]=counts["count"]/counts["group_total"]*100
    eligible_groups=counts["group_total"]>=SUPPRESSION_THRESHOLD
    facet_relationship=counts[eligible_groups].copy()
    top_groups=X_cat["grupo_vulneravel"].value_counts().head(6).index
    facet_relationship=facet_relationship[facet_relationship["grupo_vulneravel"].isin(top_groups)]
    fig=px.bar(facet_relationship,x="relacao_vitima_suspeito",y="share_pct",color="violacao",facet_row="grupo_vulneravel",barmode="stack",title="Violation composition by relationship and vulnerable group",labels={"share_pct":"Within-group share (%)"},height=1300)
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"faceted_violation_relationship_group.html",include_plotlyjs=True)
else:
    facet_relationship=pd.DataFrame()


## 8.8 Parallel categories

Parallel categories display frequent multidimensional paths. To reduce clutter and disclosure risk, category collapsing is retained and only a reproducible sample is plotted when the dataset is large.


In [9]:
PARCAT_COLUMNS=[c for c in ["grupo_vulneravel","faixa_etaria_da_vitima","genero_da_vitima","relacao_vitima_suspeito","violacao"] if c in X_cat.columns]
if len(PARCAT_COLUMNS)>=3:
    parcat_data=X_cat[PARCAT_COLUMNS].copy()
    if len(parcat_data)>SAMPLE_FOR_PLOTS:
        parcat_data=parcat_data.sample(SAMPLE_FOR_PLOTS,random_state=RANDOM_STATE)
    parcat_counts=parcat_data.groupby(PARCAT_COLUMNS).size().rename("count").reset_index()
    parcat_counts=parcat_counts[parcat_counts["count"]>=SUPPRESSION_THRESHOLD]
    fig=px.parallel_categories(parcat_counts,dimensions=PARCAT_COLUMNS,color="count",color_continuous_scale="Blues",title="Frequent multidimensional category paths")
    fig.update_layout(template=PLOTLY_TEMPLATE,height=750)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"parallel_categories_profiles.html",include_plotlyjs=True)
else:
    parcat_counts=pd.DataFrame()


## 8.9 Indicator matrix for MCA

The indicator matrix contains one binary column for each retained category. In MCA, every categorical variable contributes a set of mutually exclusive indicators. Variables with many levels can dominate interpretation, so category counts are constrained before encoding.


In [10]:
try:
    encoder=OneHotEncoder(handle_unknown="ignore",sparse_output=True,dtype=float)
except TypeError:
    encoder=OneHotEncoder(handle_unknown="ignore",sparse=True,dtype=float)

Z=encoder.fit_transform(X_cat)
indicator_names=encoder.get_feature_names_out(SELECTED_FEATURES)
indicator_inventory=pd.DataFrame({"indicator":indicator_names})
indicator_inventory["variable"]=indicator_inventory["indicator"].map(lambda x: next((c for c in SELECTED_FEATURES if x.startswith(c+"_")),"unknown"))
print("Indicator matrix:",Z.shape)
display(indicator_inventory.head(50))


Indicator matrix: (8401, 163)


,indicator,variable
0,violacao_<MISSING>,violacao
1,violacao_INTEGRIDADE>FÍSICA>EXPOSIÇÃO DE RISCO À SAÚDE,violacao
2,violacao_INTEGRIDADE>FÍSICA>MAUS TRATOS,violacao
3,violacao_INTEGRIDADE>NEGLIGÊNCIA,violacao
4,violacao_INTEGRIDADE>PSÍQUICA>AMEAÇA OU COAÇÃO,violacao
5,violacao_INTEGRIDADE>PSÍQUICA>CONSTRANGIMENTO,violacao
6,violacao_INTEGRIDADE>PSÍQUICA>EXPOSIÇÃO,violacao
7,violacao_INTEGRIDADE>PSÍQUICA>TORTURA PSÍQUICA,violacao
8,violacao_LIBERDADE>DIREITOS INDIVIDUAIS>SEQUESTRO,violacao
9,violacao_LIBERDADE>DIREITOS INDIVIDUAIS>TRÁFICO DE CRIANÇAS - INTERNACIONAL,violacao


## 8.10 Multiple Correspondence Analysis

The implementation uses correspondence analysis of the complete disjunctive table:

1. Divide the indicator matrix by the grand total.
2. Center it against the independence model.
3. Standardize by row and column masses.
4. Apply randomized singular-value decomposition.

The coordinates are exploratory. Raw MCA inertia in high-dimensional categorical data can be diffuse, so component interpretation should emphasize category contributions and stability rather than a single variance threshold.


In [11]:
def run_mca(indicator_matrix,n_components=10,random_state=42):
    from scipy import sparse
    X=indicator_matrix.tocsr().astype(float)
    grand_total=X.sum()
    P=X/grand_total
    r=np.asarray(P.sum(axis=1)).ravel()
    c=np.asarray(P.sum(axis=0)).ravel()
    expected=sparse.csr_matrix(r[:,None]@c[None,:])
    Dr_inv_sqrt=sparse.diags(1/np.sqrt(np.where(r>0,r,1)))
    Dc_inv_sqrt=sparse.diags(1/np.sqrt(np.where(c>0,c,1)))
    S=Dr_inv_sqrt@(P-expected)@Dc_inv_sqrt
    k=min(n_components,min(S.shape)-1)
    U,s,Vt=randomized_svd(S,n_components=k,random_state=random_state)
    row_coordinates=(Dr_inv_sqrt@U) * s
    column_coordinates=(Dc_inv_sqrt@Vt.T) * s
    eigenvalues=s**2
    explained=eigenvalues/eigenvalues.sum()*100
    return row_coordinates,column_coordinates,eigenvalues,explained,c

mca_rows,mca_categories,mca_eigenvalues,mca_explained,mca_column_masses=run_mca(Z,MAX_MCA_COMPONENTS,RANDOM_STATE)
component_names=[f"Dimension {i+1}" for i in range(mca_rows.shape[1])]
mca_row_coordinates=pd.DataFrame(mca_rows,index=X_cat.index,columns=component_names)
mca_category_coordinates=pd.DataFrame(mca_categories,index=indicator_names,columns=component_names)
mca_inertia=pd.DataFrame({"dimension":component_names,"eigenvalue":mca_eigenvalues,"explained_inertia_pct":mca_explained,"cumulative_pct":np.cumsum(mca_explained)})
display(mca_inertia)

fig=px.bar(mca_inertia,x="dimension",y="explained_inertia_pct",text="explained_inertia_pct",title="MCA scree plot",labels={"explained_inertia_pct":"Explained inertia (%)"})
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"mca_scree_plot.html",include_plotlyjs=True)


,dimension,eigenvalue,explained_inertia_pct,cumulative_pct
0,Dimension 1,0.295523,15.896937,15.896937
1,Dimension 2,0.227709,12.249066,28.146002
2,Dimension 3,0.198826,10.695369,38.841371
3,Dimension 4,0.190606,10.253204,49.094575
4,Dimension 5,0.176617,9.500709,58.595284
5,Dimension 6,0.169551,9.120605,67.715889
6,Dimension 7,0.159562,8.583266,76.299156
7,Dimension 8,0.154722,8.322873,84.622028
8,Dimension 9,0.145593,7.831844,92.453873
9,Dimension 10,0.140282,7.546127,100.000000


## 8.11 Category contributions and MCA maps

Approximate category contributions are based on category mass multiplied by squared principal coordinates, normalized within each dimension. Categories with high contributions help define the dimension. Rare or missing categories can be influential and should be interpreted cautiously.


In [12]:
contribution_rows=[]
for j,dimension in enumerate(component_names):
    raw=mca_column_masses*(mca_category_coordinates[dimension].to_numpy()**2)
    contribution=raw/raw.sum()*100 if raw.sum()>0 else raw
    for indicator,coord,contrib in zip(indicator_names,mca_category_coordinates[dimension],contribution):
        contribution_rows.append({"dimension":dimension,"indicator":indicator,"coordinate":coord,"contribution_pct":contrib})
mca_contributions=pd.DataFrame(contribution_rows)
top_contributions=mca_contributions.sort_values(["dimension","contribution_pct"],ascending=[True,False]).groupby("dimension").head(15)
display(top_contributions)

category_map=mca_category_coordinates[["Dimension 1","Dimension 2"]].copy().reset_index(names="indicator")
category_map=category_map.merge(mca_contributions[mca_contributions["dimension"]=="Dimension 1"][["indicator","contribution_pct"]].rename(columns={"contribution_pct":"contribution_dim1"}),on="indicator",how="left")
category_map=category_map.merge(mca_contributions[mca_contributions["dimension"]=="Dimension 2"][["indicator","contribution_pct"]].rename(columns={"contribution_pct":"contribution_dim2"}),on="indicator",how="left")
category_map["max_contribution"]=category_map[["contribution_dim1","contribution_dim2"]].max(axis=1)
category_map["variable"]=category_map["indicator"].map(lambda x: next((c for c in SELECTED_FEATURES if x.startswith(c+"_")),"unknown"))
plot_categories=category_map.nlargest(min(60,len(category_map)),"max_contribution")
fig=px.scatter(plot_categories,x="Dimension 1",y="Dimension 2",color="variable",size="max_contribution",hover_name="indicator",title="MCA category map: highest-contributing categories")
fig.add_hline(y=0,line_color="gray",line_width=1)
fig.add_vline(x=0,line_color="gray",line_width=1)
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"mca_category_map.html",include_plotlyjs=True)


,dimension,indicator,coordinate,contribution_pct
108,Dimension 1,genero_do_suspeito_<MISSING>,1.442030,9.435097
126,Dimension 1,faixa_etaria_do_suspeito_<MISSING>,1.093739,9.103992
141,Dimension 1,suspeito_preso_<MISSING>,0.950298,7.602279
140,Dimension 1,natureza_juridica_do_suspeito_ÓRGÃO PÚBLICO,1.587073,6.445348
129,Dimension 1,raca_cor_do_suspeito_<MISSING>,0.788577,6.023212
143,Dimension 1,suspeito_preso_NÃO,-0.806656,4.811709
46,Dimension 1,cenario_da_violacao_OUTROS,0.898324,3.290603
131,Dimension 1,raca_cor_do_suspeito_BRANCA,-0.838861,2.859386
34,Dimension 1,canal_de_atendimento_WHATSAPP,0.896301,2.845630
138,Dimension 1,natureza_juridica_do_suspeito_PESSOA FÍSICA,-0.418256,2.703300


## 8.12 Record map in MCA space

A reproducible sample is plotted for readability. Coloring by violation or vulnerable group helps assess whether known categories align with the first two MCA dimensions.


In [13]:
record_map=mca_row_coordinates[["Dimension 1","Dimension 2"]].copy()
for c in ["violacao","grupo_vulneravel","genero_da_vitima","uf"]:
    if c in X_cat.columns: record_map[c]=X_cat[c]
if len(record_map)>SAMPLE_FOR_PLOTS:
    record_map_plot=record_map.sample(SAMPLE_FOR_PLOTS,random_state=RANDOM_STATE)
else:
    record_map_plot=record_map
color_column="violacao" if "violacao" in record_map_plot.columns else None
fig=px.scatter(record_map_plot,x="Dimension 1",y="Dimension 2",color=color_column,opacity=.55,title="Records in the first two MCA dimensions",hover_data=[c for c in ["grupo_vulneravel","genero_da_vitima","uf"] if c in record_map_plot])
fig.update_traces(marker=dict(size=6))
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"mca_record_map.html",include_plotlyjs=True)


## 8.13 Select the number of exploratory clusters

K-means is fitted to the first MCA coordinates. Candidate solutions are compared using silhouette score, inertia, minimum cluster size, and minimum cluster share. A high silhouette score alone is insufficient when it creates tiny or disclosure-prone clusters.


In [14]:
n_cluster_dimensions=min(5,mca_row_coordinates.shape[1])
cluster_input=mca_row_coordinates.iloc[:,:n_cluster_dimensions].to_numpy()
cluster_evaluation=[]
cluster_models={}
for k in CLUSTER_K_RANGE:
    if k>=len(cluster_input): continue
    model=KMeans(n_clusters=k,n_init=20,random_state=RANDOM_STATE)
    labels=model.fit_predict(cluster_input)
    sizes=pd.Series(labels).value_counts()
    silhouette=silhouette_score(cluster_input,labels) if len(np.unique(labels))>1 else np.nan
    cluster_evaluation.append({
        "k":k,"silhouette":silhouette,"inertia":model.inertia_,
        "minimum_cluster_n":int(sizes.min()),"minimum_cluster_pct":round(sizes.min()/len(labels)*100,2)
    })
    cluster_models[k]=model
cluster_evaluation=pd.DataFrame(cluster_evaluation)
display(cluster_evaluation)

fig=make_subplots(rows=1,cols=2,subplot_titles=("Silhouette score","K-means inertia"))
fig.add_trace(go.Scatter(x=cluster_evaluation["k"],y=cluster_evaluation["silhouette"],mode="lines+markers",name="Silhouette"),row=1,col=1)
fig.add_trace(go.Scatter(x=cluster_evaluation["k"],y=cluster_evaluation["inertia"],mode="lines+markers",name="Inertia"),row=1,col=2)
fig.update_layout(template=PLOTLY_TEMPLATE,title="Candidate cluster solutions",showlegend=False)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"cluster_selection.html",include_plotlyjs=True)

eligible=cluster_evaluation[(cluster_evaluation["minimum_cluster_n"]>=SUPPRESSION_THRESHOLD)&(cluster_evaluation["minimum_cluster_pct"]>=1.0)]
if FINAL_N_CLUSTERS is None:
    selected_k=int(eligible.sort_values("silhouette",ascending=False).iloc[0]["k"] if not eligible.empty else cluster_evaluation.sort_values("silhouette",ascending=False).iloc[0]["k"])
else:
    selected_k=int(FINAL_N_CLUSTERS)
print("Selected k:",selected_k)


,k,silhouette,inertia,minimum_cluster_n,minimum_cluster_pct
0,2,0.265058,7133.213022,3033,36.10
1,3,0.294585,6044.268593,416,4.95
2,4,0.323063,5023.767680,175,2.08
3,5,0.324645,4113.184061,175,2.08
4,6,0.345315,3453.193377,175,2.08
5,7,0.339502,3068.052384,175,2.08
6,8,0.340731,2714.082868,175,2.08


Selected k: 6


## 8.14 Fit and visualize the exploratory cluster solution


In [15]:
final_model=cluster_models.get(selected_k)
if final_model is None:
    final_model=KMeans(n_clusters=selected_k,n_init=20,random_state=RANDOM_STATE).fit(cluster_input)
cluster_labels=final_model.labels_
cluster_assignments=pd.DataFrame(index=X_cat.index)
cluster_assignments["cluster"]=(cluster_labels+1).astype(int)
cluster_sizes=cluster_assignments["cluster"].value_counts().sort_index().rename_axis("cluster").reset_index(name="n")
cluster_sizes["percent"]=(cluster_sizes["n"]/len(cluster_assignments)*100).round(2)
display(cluster_sizes)

cluster_plot=mca_row_coordinates[["Dimension 1","Dimension 2"]].copy()
cluster_plot["cluster"]=(cluster_labels+1).astype(str)
if len(cluster_plot)>SAMPLE_FOR_PLOTS:
    cluster_plot=cluster_plot.sample(SAMPLE_FOR_PLOTS,random_state=RANDOM_STATE)
fig=px.scatter(cluster_plot,x="Dimension 1",y="Dimension 2",color="cluster",opacity=.6,title=f"Exploratory K-means clusters in MCA space, k={selected_k}")
fig.update_traces(marker=dict(size=6))
fig.update_layout(template=PLOTLY_TEMPLATE)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"mca_clusters.html",include_plotlyjs=True)


,cluster,n,percent
0,1,1970,23.45
1,2,3204,38.14
2,3,1139,13.56
3,4,175,2.08
4,5,1429,17.01
5,6,484,5.76


## 8.15 Cluster stability by repeated initialization

Adjusted Rand Index compares the selected labels with labels from different random seeds. This evaluates initialization sensitivity, not stability to sampling, variables, missingness coding, or category thresholds.


In [16]:
stability_rows=[]
reference_labels=cluster_labels
for seed in range(10):
    model=KMeans(n_clusters=selected_k,n_init=20,random_state=seed)
    labels=model.fit_predict(cluster_input)
    stability_rows.append({"seed":seed,"adjusted_rand_index":adjusted_rand_score(reference_labels,labels)})
cluster_stability=pd.DataFrame(stability_rows)
display(cluster_stability)


,seed,adjusted_rand_index
0,0,0.999651
1,1,0.999303
2,2,0.999303
3,3,0.999047
4,4,0.999047
5,5,0.999047
6,6,0.999047
7,7,0.999047
8,8,0.999303
9,9,0.999303


## 8.16 Profile clusters with category lift

Lift compares the category share inside a cluster with its share in the full analytical table:

`lift = cluster category proportion / overall category proportion`

Values above 1 indicate overrepresentation. The profile table requires both a minimum count and a minimum cluster share to reduce unstable interpretations.


In [17]:
profile_data=X_cat.copy()
profile_data["cluster"]=(cluster_labels+1).astype(int)
cluster_profile_rows=[]
for cluster in sorted(profile_data["cluster"].unique()):
    subset=profile_data[profile_data["cluster"]==cluster]
    for column in SELECTED_FEATURES:
        overall=profile_data[column].value_counts(normalize=True)
        within=subset[column].value_counts()
        for category,count in within.items():
            cluster_share=count/len(subset)
            overall_share=overall.get(category,np.nan)
            lift=cluster_share/overall_share if overall_share and pd.notna(overall_share) else np.nan
            cluster_profile_rows.append({
                "cluster":cluster,"column":column,"category":category,
                "count":int(count),"cluster_share_pct":round(cluster_share*100,2),
                "overall_share_pct":round(overall_share*100,2),"lift":round(lift,3)
            })
cluster_profiles=pd.DataFrame(cluster_profile_rows)
cluster_profile_highlights=cluster_profiles[
    (cluster_profiles["count"]>=SUPPRESSION_THRESHOLD)
    & (cluster_profiles["cluster_share_pct"]>=10)
].sort_values(["cluster","lift"],ascending=[True,False]).groupby("cluster").head(20)
display(cluster_profile_highlights)


,cluster,column,category,count,cluster_share_pct,overall_share_pct,lift
93,1,natureza_juridica_do_suspeito,ÓRGÃO PÚBLICO,990,50.25,12.86,3.909
78,1,raca_cor_da_vitima,AMARELA,203,10.30,3.00,3.435
86,1,genero_do_suspeito,<MISSING>,1509,76.60,22.79,3.360
100,1,uf,CE,244,12.39,3.87,3.202
95,1,natureza_juridica_do_suspeito,PESSOA JURÍDICA DE DIREITO PRIVADO,322,16.35,5.99,2.730
91,1,faixa_etaria_do_suspeito,<MISSING>,1970,100.00,38.23,2.616
24,1,canal_de_atendimento,WHATSAPP,830,42.13,17.80,2.368
97,1,suspeito_preso,<MISSING>,1970,100.00,42.29,2.364
62,1,faixa_etaria_da_vitima,15 ANOS,265,13.45,6.15,2.186
92,1,raca_cor_do_suspeito,<MISSING>,1970,100.00,48.66,2.055


### Interactive cluster profile


In [18]:
if not cluster_profile_highlights.empty:
    profile_plot=cluster_profile_highlights.copy()
    profile_plot["label"]=profile_plot["column"]+" = "+profile_plot["category"].astype(str)
    fig=px.bar(profile_plot,x="lift",y="label",color="cluster",facet_col="cluster",facet_col_wrap=2,orientation="h",hover_data=["count","cluster_share_pct","overall_share_pct"],title="Highest category lifts by cluster",height=1100)
    fig.add_vline(x=1,line_dash="dash",line_color="gray")
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"cluster_category_lifts.html",include_plotlyjs=True)


## 8.17 Cluster context: time, geography, victim quantity, and missingness

These contextual comparisons help determine whether clusters mainly reflect substantive category combinations, particular periods, geographic concentration, record size, or missing-data patterns.


In [19]:
context=df_analysis.loc[X_cat.index].copy()
context["cluster"]=(cluster_labels+1).astype(int)
if DATE_COLUMN in context.columns:
    context["year_month"]=context[DATE_COLUMN].dt.to_period("M").astype("string")
if "sl_quantidade_vitimas" in context.columns:
    context["sl_quantidade_vitimas"]=pd.to_numeric(context["sl_quantidade_vitimas"],errors="coerce")
context["selected_feature_missing_n"]=df_analysis.loc[X_cat.index,SELECTED_FEATURES].isna().sum(axis=1)

cluster_context_rows=[]
for cluster,g in context.groupby("cluster"):
    row={
        "cluster":cluster,"n":len(g),
        "mean_selected_feature_missing":g["selected_feature_missing_n"].mean()
    }
    if "sl_quantidade_vitimas" in g:
        row.update({
            "victim_quantity_median":g["sl_quantidade_vitimas"].median(),
            "victim_quantity_mean":g["sl_quantidade_vitimas"].mean()
        })
    if DATE_COLUMN in g:
        row.update({"date_min":g[DATE_COLUMN].min(),"date_max":g[DATE_COLUMN].max()})
    cluster_context_rows.append(row)
cluster_context=pd.DataFrame(cluster_context_rows)
display(cluster_context)

if "year_month" in context.columns:
    cluster_time=context.groupby(["year_month","cluster"]).size().rename("count").reset_index()
    cluster_time["monthly_total"]=cluster_time.groupby("year_month")["count"].transform("sum")
    cluster_time["share_pct"]=cluster_time["count"]/cluster_time["monthly_total"]*100
    fig=px.area(cluster_time,x="year_month",y="share_pct",color="cluster",title="Monthly cluster composition",labels={"share_pct":"Monthly share (%)"})
    fig.update_layout(template=PLOTLY_TEMPLATE,yaxis_range=[0,100])
    fig.show()
    if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"cluster_composition_over_time.html",include_plotlyjs=True)
else:
    cluster_time=pd.DataFrame()


,cluster,n,mean_selected_feature_missing,victim_quantity_median,victim_quantity_mean,date_min,date_max
0,1,1970,4.540102,1.0,3.155330,2021-01-26 17:03:00,2025-12-20 19:31:39
1,2,3204,1.162921,1.0,3.192572,2021-01-08 07:02:00,2025-12-30 15:31:58
2,3,1139,0.800702,1.0,1.248464,2021-05-02 22:16:00,2025-12-20 19:31:39
3,4,175,0.800000,2.0,2.000000,2025-12-11 10:44:47,2025-12-11 10:44:47
4,5,1429,2.077677,1.0,1.521344,2021-11-22 21:02:00,2025-12-28 14:53:31
5,6,484,6.320248,10.0,14.935950,2021-01-05 08:28:00,2023-12-30 11:12:32


## 8.18 Missingness sensitivity

Because missing values are included as categories, clusters may be driven by documentation completeness. This section reports each cluster's missing-category share and flags profiles dominated by `<MISSING>`.


In [20]:
missing_profile=cluster_profiles[cluster_profiles["category"]==MISSING_LABEL].copy()
missing_profile=missing_profile.sort_values(["cluster","cluster_share_pct"],ascending=[True,False])
display(missing_profile)

cluster_missing_summary=missing_profile.groupby("cluster").agg(
    variables_with_missing_category=("column","size"),
    mean_missing_category_share=("cluster_share_pct","mean"),
    max_missing_category_share=("cluster_share_pct","max")
).reset_index() if not missing_profile.empty else pd.DataFrame()
display(cluster_missing_summary)


,cluster,column,category,count,cluster_share_pct,overall_share_pct,lift
91,1,faixa_etaria_do_suspeito,<MISSING>,1970,100.00,38.23,2.616
92,1,raca_cor_do_suspeito,<MISSING>,1970,100.00,48.66,2.055
97,1,suspeito_preso,<MISSING>,1970,100.00,42.29,2.364
86,1,genero_do_suspeito,<MISSING>,1509,76.60,22.79,3.360
83,1,orientacao_sexual_da_vitima,<MISSING>,809,41.07,39.47,1.040
56,1,genero_da_vitima,<MISSING>,537,27.26,17.71,1.539
80,1,raca_cor_da_vitima,<MISSING>,77,3.91,9.63,0.406
106,1,uf,<MISSING>,71,3.60,8.65,0.416
14,1,violacao,<MISSING>,28,1.42,2.26,0.628
74,1,faixa_etaria_da_vitima,<MISSING>,3,0.15,4.82,0.032


,cluster,variables_with_missing_category,mean_missing_category_share,max_missing_category_share
0,1,10,45.4010,100.00
1,2,10,11.5870,55.15
2,3,7,11.4400,28.71
3,4,4,20.0000,20.00
4,5,8,25.9725,62.28
5,6,10,62.5000,95.04


## 8.19 Cluster assignment disclosure controls

Record-level cluster assignments can be sensitive. The exported assignment file contains the source index, optional `hash`, and cluster label for restricted analytical use. Public reporting should use aggregated and suppressed tables only.


In [21]:
restricted_assignments=cluster_assignments.reset_index(names="source_index")
if IDENTIFIER in df_analysis.columns:
    restricted_assignments[IDENTIFIER]=df_analysis.loc[X_cat.index,IDENTIFIER].to_numpy()
restricted_assignments["public_release_allowed"]=False
aggregated_cluster_release=cluster_sizes.copy()
aggregated_cluster_release["suppressed"]=aggregated_cluster_release["n"]<SUPPRESSION_THRESHOLD
aggregated_cluster_release["display_n"]=aggregated_cluster_release["n"].where(~aggregated_cluster_release["suppressed"],pd.NA)
display(aggregated_cluster_release)


,cluster,n,percent,suppressed,display_n
0,1,1970,23.45,False,1970
1,2,3204,38.14,False,3204
2,3,1139,13.56,False,1139
3,4,175,2.08,False,175
4,5,1429,17.01,False,1429
5,6,484,5.76,False,484


## 8.20 Automated multivariate findings register

The register highlights strong variable associations, leading MCA contributors, cluster quality, instability, missingness-driven profiles, and small clusters. These findings require substantive interpretation.


In [22]:
finding_rows=[]
for _,r in high_association_pairs[high_association_pairs["cramers_v"]>=.50].iterrows():
    finding_rows.append({
        "finding_type":"strong_categorical_association","scope":f"{r['variable_1']} × {r['variable_2']}",
        "finding":f"Bias-corrected Cramér's V={r['cramers_v']:.3f}",
        "interpretation":"Review conceptual overlap, hierarchy, and redundant influence before clustering interpretation."
    })
for dimension,g in top_contributions.groupby("dimension"):
    leaders=g.head(5)["indicator"].tolist()
    finding_rows.append({
        "finding_type":"mca_dimension","scope":dimension,
        "finding":"Highest contributors: "+"; ".join(leaders),
        "interpretation":"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
    })
selected_eval=cluster_evaluation[cluster_evaluation["k"]==selected_k].iloc[0]
finding_rows.append({
    "finding_type":"cluster_solution","scope":f"k={selected_k}",
    "finding":f"Silhouette={selected_eval['silhouette']:.3f}; smallest cluster={int(selected_eval['minimum_cluster_n'])}",
    "interpretation":"Treat clusters as exploratory summaries and compare with alternative k and feature sets."
})
if cluster_stability["adjusted_rand_index"].mean()<.80:
    finding_rows.append({
        "finding_type":"cluster_instability","scope":"initialization",
        "finding":f"Mean adjusted Rand index={cluster_stability['adjusted_rand_index'].mean():.3f}",
        "interpretation":"Cluster assignments are sensitive to initialization; avoid stable typology claims."
    })
for _,r in cluster_missing_summary.iterrows() if not cluster_missing_summary.empty else []:
    if r["max_missing_category_share"]>=50:
        finding_rows.append({
            "finding_type":"missingness_driven_cluster","scope":f"cluster {int(r['cluster'])}",
            "finding":f"At least one variable is missing in {r['max_missing_category_share']:.1f}% of the cluster",
            "interpretation":"Interpret as a possible documentation pattern rather than a substantive profile."
        })
multivariate_findings=pd.DataFrame(finding_rows)
display(multivariate_findings)


,finding_type,scope,finding,interpretation
0,mca_dimension,Dimension 1,Highest contributors: genero_do_suspeito_<MISSING>; faixa_etaria_do_suspeito_<MISSING>; suspeito_preso_<MISSING>; natureza_juridica_do_suspeito_ÓRGÃO PÚBLICO; raca_cor_do_suspe...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
1,mca_dimension,Dimension 10,Highest contributors: orientacao_sexual_da_vitima_HOMOSSEXUAL.GAY; genero_do_suspeito_INTERSEXO; grupo_vulneravel_VIOLÊNCIA CONTRA POPULAÇÃO LGBTQIA+; suspeito_preso_PENA (RECL...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
2,mca_dimension,Dimension 2,Highest contributors: orientacao_sexual_da_vitima_<MISSING>; grupo_vulneravel_VIOLÊNCIA CONTRA A MULHER; faixa_etaria_da_vitima_<MISSING>; raca_cor_da_vitima_<MISSING>; genero_...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
3,mca_dimension,Dimension 3,Highest contributors: relacao_vitima_suspeito_CUNHADO(A); frequencia_ÚNICA OCORRÊNCIA; cenario_da_violacao_CASA DE FAMILIARES; raca_cor_da_vitima_BRANCA; orientacao_sexual_da_v...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
4,mca_dimension,Dimension 4,Highest contributors: cenario_da_violacao_CASA DE FAMILIARES; raca_cor_da_vitima_<MISSING>; denuncia_emergencial_RISCO IMINENTE DE MORTE DA VÍTIMA; relacao_vitima_suspeito_OUTR...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
5,mca_dimension,Dimension 5,Highest contributors: suspeito_preso_NÃO INFORMADO; relacao_vitima_suspeito_CUNHADO(A); faixa_etaria_do_suspeito_NÃO INFORMADO; faixa_etaria_da_vitima_30 A 34 ANOS; frequencia_...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
6,mca_dimension,Dimension 6,Highest contributors: cenario_da_violacao_ESTABELECIMENTO COMERCIAL; uf_MS; faixa_etaria_da_vitima_<MISSING>; uf_DENUNCIANTE NÃO SOUBE INFORMAR; uf_<MISSING>,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
7,mca_dimension,Dimension 7,Highest contributors: uf_MS; cenario_da_violacao_ESTABELECIMENTO COMERCIAL; uf_CE; faixa_etaria_da_vitima_CRIANÇA/ADOLESCENTE IDADE NÃO INFORMADA; raca_cor_da_vitima_INDÍGENA,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."
8,mca_dimension,Dimension 8,"Highest contributors: uf_RN; cenario_da_violacao_EVENTO PÚBLICO; grupo_vulneravel_VIOLÊNCIA CONTRA CIDADÃO, FAMÍLIA OU COMUNIDADE; genero_da_vitima_NÃO INFORMADO; cenario_da_vi...","Interpret the dimension from opposing category coordinates and contributions, not labels alone."
9,mca_dimension,Dimension 9,Highest contributors: raca_cor_da_vitima_INDÍGENA; cenario_da_violacao_VIA PÚBLICA; relacao_vitima_suspeito_NÃO SABE INFORMAR; canal_de_atendimento_E-MAIL; raca_cor_da_vitima_N...,"Interpret the dimension from opposing category coordinates and contributions, not labels alone."


## 8.21 Export tables, figures, coordinates, and metadata


In [23]:
exports={
    "feature_inventory.csv":feature_inventory,
    "category_reduction.csv":category_reduction,
    "cramers_v_matrix.csv":association_matrix.reset_index(names="variable"),
    "association_pairs.csv":high_association_pairs,
    "facet_violation_age_gender.csv":facet_violation,
    "facet_violation_relationship_group.csv":facet_relationship,
    "parallel_category_paths.csv":parcat_counts,
    "indicator_inventory.csv":indicator_inventory,
    "mca_inertia.csv":mca_inertia,
    "mca_category_coordinates.csv":mca_category_coordinates.reset_index(names="indicator"),
    "mca_category_contributions.csv":mca_contributions,
    "mca_top_contributions.csv":top_contributions,
    "cluster_evaluation.csv":cluster_evaluation,
    "cluster_sizes.csv":cluster_sizes,
    "cluster_stability.csv":cluster_stability,
    "cluster_profiles.csv":cluster_profiles,
    "cluster_profile_highlights.csv":cluster_profile_highlights,
    "cluster_context.csv":cluster_context,
    "cluster_time.csv":cluster_time,
    "cluster_missing_profile.csv":missing_profile,
    "cluster_missing_summary.csv":cluster_missing_summary,
    "aggregated_cluster_release.csv":aggregated_cluster_release,
    "multivariate_findings.csv":multivariate_findings,
}
for filename,table in exports.items():
    table.to_csv(TABLE_DIR/filename,index=False,encoding="utf-8-sig")

# Restricted outputs: keep access-controlled.
restricted_assignments.to_csv(TABLE_DIR/"RESTRICTED_cluster_assignments.csv",index=False,encoding="utf-8-sig")
mca_row_coordinates.reset_index(names="source_index").to_csv(TABLE_DIR/"RESTRICTED_mca_row_coordinates.csv",index=False,encoding="utf-8-sig")

figure_index=pd.DataFrame({"figure_file":sorted(p.name for p in FIGURE_DIR.glob("*.html"))})
figure_index.to_csv(OUTPUT_DIR/"figure_index.csv",index=False,encoding="utf-8-sig")
metadata={
    "data_source":str(DATA_SOURCE),"counting_mode":COUNTING_MODE,
    "analytical_unit":unit_label,"analytical_rows":int(len(df_analysis)),
    "selected_features":SELECTED_FEATURES,"missing_value_strategy":"explicit category",
    "minimum_category_count":MIN_CATEGORY_COUNT,
    "maximum_levels_per_variable":MAX_LEVELS_PER_VARIABLE,
    "mca_components":int(mca_row_coordinates.shape[1]),
    "cluster_dimensions_used":int(n_cluster_dimensions),
    "selected_clusters":int(selected_k),"random_state":RANDOM_STATE,
    "suppression_threshold":SUPPRESSION_THRESHOLD,
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Tables:",TABLE_DIR.resolve())
print("Interactive figures:",FIGURE_DIR.resolve())


Tables: C:\Projetos\eda-disque100-completo\eda\outputs\step_08_multivariate_exploration\tables
Interactive figures: C:\Projetos\eda-disque100-completo\eda\outputs\step_08_multivariate_exploration\figures


## 8.22 Interpretation checklist

Before reporting multivariate findings:

1. Confirm whether the unit is a record or unique complaint.
2. Explain why each variable was included and which variables were excluded.
3. Report category-collapsing and missingness rules.
4. Review high Cramér's V pairs for redundancy or hierarchy.
5. Interpret MCA dimensions from coordinates and contributions on both sides of each axis.
6. Do not assign substantive names to clusters until profiles, missingness, time, and geography are jointly reviewed.
7. Report silhouette score, cluster sizes, and stability results.
8. Test sensitivity to the number of MCA dimensions, category threshold, feature set, and counting mode.
9. Treat missingness-dominated clusters as documentation patterns.
10. Protect restricted cluster assignments and suppress small aggregate cells.
11. Avoid causal, diagnostic, profiling, or predictive-risk claims.
12. Validate any proposed profile with domain experts and an independent dataset or time period.

### Recommended reporting language

“Multiple correspondence analysis summarized associations among selected categorical attributes of recorded hotline observations. Exploratory K-means clustering of MCA coordinates identified candidate groups of records with different category compositions. These groups are descriptive summaries of the administrative data and may reflect reporting practices, form structure, missingness, and repeated-record processes. They are not interpreted as natural victim or suspect typologies.”

## Next step

**Step 9: Hypothesis Generation and Confirmation Plan** should convert robust descriptive patterns into explicit hypotheses, define estimands and denominators, distinguish exploratory from confirmatory analyses, specify statistical tests and sensitivity checks, and prevent reuse of the same data for unrestricted discovery and confirmation.
